In [20]:
#!pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "city_temperature.csv"

city_temperature = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sudalairajkumar/daily-temperature-of-major-cities",
  file_path,
)

print("First 5 records:", city_temperature.head())

/tmp/ipykernel_504/1269568289.py:7: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  city_temperature = kagglehub.load_dataset(


Using Colab cache for faster access to the 'daily-temperature-of-major-cities' dataset.


/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:91: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


First 5 records:    Region  Country State     City  Month  Day  Year  AvgTemperature
0  Africa  Algeria   NaN  Algiers      1    1  1995            64.2
1  Africa  Algeria   NaN  Algiers      1    2  1995            49.4
2  Africa  Algeria   NaN  Algiers      1    3  1995            48.8
3  Africa  Algeria   NaN  Algiers      1    4  1995            46.4
4  Africa  Algeria   NaN  Algiers      1    5  1995            47.9


In [21]:
#!pip install duckdb
import duckdb

In [22]:
def exec_query_duckdb(query):
  return duckdb.query(query).to_df()

df_1 = exec_query_duckdb("""
SELECT * FROM 'city_temperature' LIMIT 10
""")
print(df_1)

   Region  Country State     City  Month  Day  Year  AvgTemperature
0  Africa  Algeria  None  Algiers      1    1  1995            64.2
1  Africa  Algeria  None  Algiers      1    2  1995            49.4
2  Africa  Algeria  None  Algiers      1    3  1995            48.8
3  Africa  Algeria  None  Algiers      1    4  1995            46.4
4  Africa  Algeria  None  Algiers      1    5  1995            47.9
5  Africa  Algeria  None  Algiers      1    6  1995            48.7
6  Africa  Algeria  None  Algiers      1    7  1995            48.9
7  Africa  Algeria  None  Algiers      1    8  1995            49.1
8  Africa  Algeria  None  Algiers      1    9  1995            49.0
9  Africa  Algeria  None  Algiers      1   10  1995            51.9


# Conversão e tratamento temporal, transformando os campos de data para uma única coluna no formato apropriado (DATE), extraindo ano/mês/dia e convertendo o fuso horário caso necessário.

Verificando a existência de linhas com dia 0

In [23]:
df_4 = exec_query_duckdb("""
SELECT * FROM 'city_temperature' WHERE Day = 0
""")
print(df_4)

                              Region        Country State         City  Month  \
0                             Africa         Guinea  None      Conakry      3   
1                             Africa         Guinea  None      Conakry      3   
2                             Africa  Guinea-Bissau  None       Bissau      3   
3                             Africa         Malawi  None     Lilongwe      3   
4                             Africa        Nigeria  None        Lagos      3   
5                             Africa         Uganda  None      Kampala      3   
6                      North America         Mexico  None  Guadalajara      3   
7  South/Central America & Carribean           Cuba  None       Havana      3   

   Day  Year  AvgTemperature  
0    0  2008           -99.0  
1    0  2016           -99.0  
2    0  2008           -99.0  
3    0  2012           -99.0  
4    0  2008           -99.0  
5    0  2012           -99.0  
6    0  2012           -99.0  
7    0  2008          

Tranformando dias 0 em 1. Estou presumindo que o dia 0 apenas foi colocado no lugar do dia 1

In [24]:
df_2 = exec_query_duckdb("""
SELECT
  Region, Country, State, City, Month, if(Day = 0, 1, Day) as Day, Year, AvgTemperature
FROM
  'city_temperature'
""")
print(df_2)

                Region  Country                   State                  City  \
0               Africa  Algeria                    None               Algiers   
1               Africa  Algeria                    None               Algiers   
2               Africa  Algeria                    None               Algiers   
3               Africa  Algeria                    None               Algiers   
4               Africa  Algeria                    None               Algiers   
...                ...      ...                     ...                   ...   
2906322  North America       US  Additional Territories  San Juan Puerto Rico   
2906323  North America       US  Additional Territories  San Juan Puerto Rico   
2906324  North America       US  Additional Territories  San Juan Puerto Rico   
2906325  North America       US  Additional Territories  San Juan Puerto Rico   
2906326  North America       US  Additional Territories  San Juan Puerto Rico   

         Month  Day  Year  

Defini uma data de corte do ano 1800 para baixo, datas com menos de 1800 estão erradas
E não foi possível presumir qual era o ano correto, então foram descartadas
Datas com ano superior a 2025 são impossíveis, então também serão descartadas

Esse é o 'df_clean', o dataset limpo que será utilizado nas outras análises

In [25]:
df_clean = exec_query_duckdb("""
SELECT
  *, Make_Date(Year, Month, Day) As medition_date
FROM
  'df_2'
WHERE YEAR > 1800 AND YEAR <= 2025
""")
print(df_clean)

                Region  Country                   State                  City  \
0               Africa  Algeria                    None               Algiers   
1               Africa  Algeria                    None               Algiers   
2               Africa  Algeria                    None               Algiers   
3               Africa  Algeria                    None               Algiers   
4               Africa  Algeria                    None               Algiers   
...                ...      ...                     ...                   ...   
2905882  North America       US  Additional Territories  San Juan Puerto Rico   
2905883  North America       US  Additional Territories  San Juan Puerto Rico   
2905884  North America       US  Additional Territories  San Juan Puerto Rico   
2905885  North America       US  Additional Territories  San Juan Puerto Rico   
2905886  North America       US  Additional Territories  San Juan Puerto Rico   

         Month  Day  Year  

#Cálculo de intervalos e duração: cálculo do tempo entre dois eventos consecutivos para cada categoria, identificando lacunas e intervalos médios.

Primeiro montei uma tabela temporária com a data anterior de medição e a diferença entre essas duas datas

Depois utilizei essa tabela 'gap_analysis' selecionando registros com diferença de dias maior que 1, ou seja que possui uma lacuna na medição

Fiz a contagem, identifiquei valor minimo, maximo, média e total de dias das lacunas


In [26]:
df_3 = exec_query_duckdb("""
WITH gap_analysis AS (
  SELECT
    city, medition_date, LAG(medition_date)
  OVER (
    PARTITION BY city ORDER BY medition_date
    ) as previous_date,
  DATEDIFF(
    'day', LAG(medition_date
    )
  OVER (
    PARTITION BY city ORDER BY medition_date
    ), medition_date) as gap_days
  FROM 'df_clean'
  WHERE medition_date IS NOT NULL
)
SELECT
  city,
  COUNT(*) as total_gaps,
  MIN(gap_days) as min_gap_days,
  MAX(gap_days) as max_gap_days,
  AVG(gap_days) as avg_gap_days,
  SUM(gap_days - 1) as total_missing_days
FROM 'gap_analysis'
WHERE gap_days > 1
GROUP BY city
ORDER BY total_missing_days DESC;
""")
print(df_3)

              City  total_gaps  min_gap_days  max_gap_days  avg_gap_days  \
0          Nairobi           3             2          1097         367.0   
1         Lilongwe           2             2          1097         549.5   
2       Georgetown           1          1097          1097        1097.0   
3      Guadalajara           2             2           732         367.0   
4           Banjul           1           732           732         732.0   
5          Conakry           2             2             2           2.0   
6           Bissau           1             2             2           2.0   
7           Havana           1             2             2           2.0   
8   Port au Prince           1             2             2           2.0   
9           Munich           1             2             2           2.0   
10         Kampala           1             2             2           2.0   
11           Lagos           1             2             2           2.0   
12         C

# Integração de múltiplas granularidades: integrei dados semanais e mensais e criei agregações em diferentes níveis temporais (semana, mês, trimestre).

Fiz agregações utilizando janelas com o 'OVER', fiz a média, máximo e mínimo das medições por mes, semana e dia

In [27]:
df_6 = exec_query_duckdb("""
SELECT
  city,
  country,
  medition_date,
  EXTRACT(WEEK FROM medition_date) as week,
  AVG(AVG(AvgTemperature)) OVER (
    PARTITION BY city, year, month
    ) as month_avg_temp,
  MAX(MAX(AvgTemperature)) OVER (
    PARTITION BY city, year, month
    ) as month_max_temp,
  MIN(MIN(AvgTemperature)) OVER (
    PARTITION BY city, year, month
    ) as month_min_temp,
  AVG(AVG(AvgTemperature)) OVER (
    PARTITION BY city, year, EXTRACT(WEEK FROM medition_date)
    ) as week_avg_temp,
  MAX(MAX(AvgTemperature)) OVER (
    PARTITION BY city, year, EXTRACT(WEEK FROM medition_date)
    ) as week_max_temp,
  MIN(MIN(AvgTemperature)) OVER (
    PARTITION BY city, year, EXTRACT(WEEK FROM medition_date)
    ) as week_min_temp,
  AVG(AvgTemperature) as daily_avg_temp,
  MAX(AvgTemperature) as daily_max_temp,
  MIN(AvgTemperature) as daily_min_temp,
FROM df_clean
WHERE
  medition_date IS NOT NULL AND
  AvgTemperature IS NOT NULL
GROUP BY city, country, medition_date, year, month, week
ORDER BY city, medition_date;
""")
print(df_6)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            City      Country medition_date  week  month_avg_temp  \
0        Abidjan  Ivory Coast    1995-01-01    52       79.916129   
1        Abidjan  Ivory Coast    1995-01-02     1       79.916129   
2        Abidjan  Ivory Coast    1995-01-03     1       79.916129   
3        Abidjan  Ivory Coast    1995-01-04     1       79.916129   
4        Abidjan  Ivory Coast    1995-01-05     1       79.916129   
...          ...          ...           ...   ...             ...   
2847991   Zurich  Switzerland    2020-05-09    19       54.000000   
2847992   Zurich  Switzerland    2020-05-10    19       54.000000   
2847993   Zurich  Switzerland    2020-05-11    20       54.000000   
2847994   Zurich  Switzerland    2020-05-12    20       54.000000   
2847995   Zurich  Switzerland    2020-05-13    20       54.000000   

         month_max_temp  month_min_temp  week_avg_temp  week_max_temp  \
0                  84.1            73.0      79.012500           82.6   
1                  84.1  

# Tendência geral: gerei uma tabela com a média móvel de 7 e 30 dias para cada categoria, mostrando tendências e oscilações sazonais.

Utilizando o partition by e o over (como sempre ao utilizar funções de janela) fiz a média das medições de 6 dias anteriores (incluindo o atual, dão 7), fiz isso para os 30 dias também. Calculei também a diferença entre a medía móvel e o a média atual, para as duas médias móveis. E ai verifiquei, se a média móvel atual (começando 7 dias atras) é maior, menor ou igual, a anterior (começando 14 dias atrás). Assim sendo classificadas as tendências e sendo possível identificar as variações sazonais (não consegui pensar uma forma para identificar as alterações sazonais, não encontrei um valor específico)

In [28]:
df_7 = exec_query_duckdb("""
SELECT
  city,
  country,
  medition_date,
  AvgTemperature as temp_diaria,
  ROUND(AVG(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) as media_movel_7d,
  ROUND(AVG(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) as media_movel_30d,
  ROUND(AvgTemperature - AVG(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) as diferenca_7d,
  ROUND(AvgTemperature - AVG(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) as diferenca_30d,
  CASE
    WHEN AVG(AvgTemperature) OVER (
      PARTITION BY city ORDER BY medition_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
      ) > AVG(AvgTemperature) OVER (
        PARTITION BY city ORDER BY medition_date ROWS BETWEEN 13 PRECEDING AND 7 PRECEDING
        ) THEN 'CRESCENTE'
    WHEN AVG(AvgTemperature) OVER (
      PARTITION BY city ORDER BY medition_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
      ) < AVG(AvgTemperature) OVER (
        PARTITION BY city ORDER BY medition_date ROWS BETWEEN 13 PRECEDING AND 7 PRECEDING
        ) THEN 'DECRESCENTE'
    ELSE 'ESTÁVEL'
  END as tendencia_7d
  FROM df_clean
  WHERE medition_date IS NOT NULL AND AvgTemperature IS NOT NULL
  ORDER BY city, medition_date;
""")
print(df_7)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            City      Country medition_date  temp_diaria  media_movel_7d  \
0        Abidjan  Ivory Coast    1995-01-01         82.6           82.60   
1        Abidjan  Ivory Coast    1995-01-02         82.3           82.45   
2        Abidjan  Ivory Coast    1995-01-03         81.0           81.97   
3        Abidjan  Ivory Coast    1995-01-04         83.3           82.30   
4        Abidjan  Ivory Coast    1995-01-05         83.4           82.52   
...          ...          ...           ...          ...             ...   
2905882   Zurich  Switzerland    2020-05-09         67.1           57.10   
2905883   Zurich  Switzerland    2020-05-10         64.7           59.00   
2905884   Zurich  Switzerland    2020-05-11         52.0           57.91   
2905885   Zurich  Switzerland    2020-05-12         43.5           56.77   
2905886   Zurich  Switzerland    2020-05-13         44.6           55.86   

         media_movel_30d  diferenca_7d  diferenca_30d tendencia_7d  
0                 

# Variação percentual: criei um índice de evolução mês a mês e calcule a variação percentual entre os períodos.

Primeiro criei uma tabela temporaria que agrega a média de temperatura pelo mês

Depois uma que seleciona a valor da média do mês anterior

E por fim a selecao que seleciona a media do mes, media do mes anterior e a porcentagem da variação do valor

In [29]:
df_8 = exec_query_duckdb("""
WITH monthly_temperatures AS (
  SELECT
    city,
    country,
    year,
    month,
    ROUND(AVG(AvgTemperature), 2) as temp_media_mensal,
    COUNT(*) as dias_medidos
  FROM df_clean
  WHERE medition_date IS NOT NULL AND
        AvgTemperature IS NOT NULL
  GROUP BY city, country, year, month
),
monthly_variation AS (
  SELECT
    city,
    country,
    year,
    month,
    temp_media_mensal,
    LAG(temp_media_mensal) OVER (
      PARTITION BY city ORDER BY year, month
      ) as temp_mes_anterior,
    dias_medidos
  FROM monthly_temperatures
)
SELECT
  city,
  country,
  year,
  month,
  temp_media_mensal,
  temp_mes_anterior,
  ROUND(
    CASE
      WHEN temp_mes_anterior IS NOT NULL AND
           temp_mes_anterior THEN ((temp_media_mensal - temp_mes_anterior) / ABS(temp_mes_anterior)) * 100
      ELSE NULL
    END, 2) as variacao_percentual,
  dias_medidos
FROM monthly_variation
ORDER BY city, year, month;
""")

print(df_8)

          City      Country  Year  Month  temp_media_mensal  \
0      Abidjan  Ivory Coast  1995      1              79.92   
1      Abidjan  Ivory Coast  1995      2              82.61   
2      Abidjan  Ivory Coast  1995      3              82.55   
3      Abidjan  Ivory Coast  1995      4              83.32   
4      Abidjan  Ivory Coast  1995      5              82.40   
...        ...          ...   ...    ...                ...   
93750   Zurich  Switzerland  2020      1              36.68   
93751   Zurich  Switzerland  2020      2              43.17   
93752   Zurich  Switzerland  2020      3              42.65   
93753   Zurich  Switzerland  2020      4              55.08   
93754   Zurich  Switzerland  2020      5              54.00   

       temp_mes_anterior  variacao_percentual  dias_medidos  
0                    NaN                  NaN            31  
1                  79.92                 3.37            28  
2                  82.61                -0.07            

# Rolling windows: apliquei janelas deslizantes (rolling) para suavizar séries temporais e identificar padrões mais persistentes. Utilizei AVG e SUM sobre janelas de 3 e 5 períodos.

Utilizando (rows between, preceding, following) selecionei as linhas anteriores, post e atual, em 2 periodos diferentes calculando média e soma em cada periodo

In [30]:
df_9 = exec_query_duckdb("""
SELECT
  city,
  country,
  medition_date,
  AvgTemperature as temp_original,
  ROUND(AVG(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING
    ), 2) as rolling_avg_3d,
  ROUND(AVG(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 4 PRECEDING AND 4 FOLLOWING
    ), 2) as rolling_avg_5d,
  ROUND(SUM(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING
    ), 2) as rolling_sum_3d,
  ROUND(SUM(AvgTemperature) OVER (
    PARTITION BY city ORDER BY medition_date ROWS BETWEEN 4 PRECEDING AND 4 FOLLOWING
    ), 2) as rolling_sum_5d,
FROM df_clean
WHERE medition_date IS NOT NULL AND
      AvgTemperature IS NOT NULL
ORDER BY city, medition_date;
""")
print(df_9)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            City      Country medition_date  temp_original  rolling_avg_3d  \
0        Abidjan  Ivory Coast    1995-01-01           82.6           81.97   
1        Abidjan  Ivory Coast    1995-01-02           82.3           82.30   
2        Abidjan  Ivory Coast    1995-01-03           81.0           82.52   
3        Abidjan  Ivory Coast    1995-01-04           83.3           82.82   
4        Abidjan  Ivory Coast    1995-01-05           83.4           82.86   
...          ...          ...           ...            ...             ...   
2905882   Zurich  Switzerland    2020-05-09           67.1           60.58   
2905883   Zurich  Switzerland    2020-05-10           64.7           58.20   
2905884   Zurich  Switzerland    2020-05-11           52.0           54.38   
2905885   Zurich  Switzerland    2020-05-12           43.5           51.20   
2905886   Zurich  Switzerland    2020-05-13           44.6           46.70   

         rolling_avg_5d  rolling_sum_3d  rolling_sum_5d  
0    

# Valores acumulados: computei somatórios acumulados por categoria.

Criei uma tabela temporaria selecionando todas as linhas anteriores da partição por cidade e ano e somei na coluna graus_dia_acumulados_ano fiz também um contador para essa partição a coluna dias_acumulados_ano na selecao arredondei os valores e com os 2 calculei a media do ano.

In [31]:
df_10 = exec_query_duckdb("""
WITH daily_accumulated AS (
  SELECT
    city,
    country,
    medition_date,
    year,
    month,
    AvgTemperature,
    SUM(AvgTemperature) OVER (
      PARTITION BY city, year ORDER BY medition_date ROWS UNBOUNDED PRECEDING
      ) as graus_dia_acumulados_ano,
    COUNT(*) OVER (
      PARTITION BY city, year ORDER BY medition_date ROWS UNBOUNDED PRECEDING
      ) as dias_acumulados_ano
  FROM df_clean
  WHERE medition_date IS NOT NULL AND
        AvgTemperature IS NOT NULL
)
SELECT
  city,
  country,
  medition_date,
  year,
  month,
  ROUND(AvgTemperature, 2) as temperatura_dia,
  ROUND(graus_dia_acumulados_ano, 2) as graus_dia_acumulados,
  dias_acumulados_ano,
  ROUND(graus_dia_acumulados_ano / dias_acumulados_ano, 2) as media_acumulada_ano
FROM daily_accumulated
ORDER BY city, medition_date;
""")
print(df_10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            City      Country medition_date  Year  Month  temperatura_dia  \
0        Abidjan  Ivory Coast    1995-01-01  1995      1             82.6   
1        Abidjan  Ivory Coast    1995-01-02  1995      1             82.3   
2        Abidjan  Ivory Coast    1995-01-03  1995      1             81.0   
3        Abidjan  Ivory Coast    1995-01-04  1995      1             83.3   
4        Abidjan  Ivory Coast    1995-01-05  1995      1             83.4   
...          ...          ...           ...   ...    ...              ...   
2905882   Zurich  Switzerland    2020-05-09  2020      5             67.1   
2905883   Zurich  Switzerland    2020-05-10  2020      5             64.7   
2905884   Zurich  Switzerland    2020-05-11  2020      5             52.0   
2905885   Zurich  Switzerland    2020-05-12  2020      5             43.5   
2905886   Zurich  Switzerland    2020-05-13  2020      5             44.6   

         graus_dia_acumulados  dias_acumulados_ano  media_acumulada_ano  
0

#Comparação entre períodos: comparei, para cada categoria, os totais de um mês com o mesmo mês do ano anterior (YoY), e também entre meses consecutivos (MoM).

Criei uma tabela temporaria agregando por mes as medições e calculando a media no mes, e contando os registros

Encima desta, selecionei também a temperatura do mes e ano anterior (utilizando LAG e LAG(,12))

E na seleção, deixei de fora as medicoes sem temperatura do mes ou ano anterior

In [32]:
df_11 = exec_query_duckdb("""
WITH monthly_aggregated AS (
  SELECT
    city,
    country,
    year,
    month,
    ROUND(AVG(AvgTemperature), 2) as temperatura_atual,
    COUNT(*) as dias_medidos
  FROM df_clean
  WHERE medition_date IS NOT NULL AND
        AvgTemperature IS NOT NULL
  GROUP BY city, country,year, month
),
comparisons AS (
  SELECT
    *,
    LAG(temperatura_atual) OVER (
      PARTITION BY city ORDER BY year, month
      ) as temperatura_mes_anterior,
    LAG(temperatura_atual, 12) OVER (
      PARTITION BY city ORDER BY year, month
      ) as temperatura_ano_anterior
  FROM monthly_aggregated
)
SELECT
  *
FROM comparisons
WHERE temperatura_mes_anterior IS NOT NULL OR
      temperatura_ano_anterior IS NOT NULL
ORDER BY city, year, month;
""")
print(df_11)

          City      Country  Year  Month  temperatura_atual  dias_medidos  \
0      Abidjan  Ivory Coast  1995      2              82.61            28   
1      Abidjan  Ivory Coast  1995      3              82.55            31   
2      Abidjan  Ivory Coast  1995      4              83.32            30   
3      Abidjan  Ivory Coast  1995      5              82.40            31   
4      Abidjan  Ivory Coast  1995      6              80.80            30   
...        ...          ...   ...    ...                ...           ...   
93429   Zurich  Switzerland  2020      1              36.68            31   
93430   Zurich  Switzerland  2020      2              43.17            29   
93431   Zurich  Switzerland  2020      3              42.65            31   
93432   Zurich  Switzerland  2020      4              55.08            30   
93433   Zurich  Switzerland  2020      5              54.00            13   

       temperatura_mes_anterior  temperatura_ano_anterior  
0              

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=420095cc-5d9c-4dcb-9941-bb54c76e9ff9' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>